# OpenRouter LLM API 실습

이 노트북은 **Google Colab**에서 OpenRouter가 제공하는 LLM을 Python으로 사용하는 방법을 실습하기 위한 자료입니다.

여기서는 OpenRouter 전용 SDK가 아니라 **OpenAI Python SDK**를 사용합니다. OpenRouter API가 OpenAI SDK와 호환되기 때문에 `base_url`만 OpenRouter 주소로 지정하면 여러 회사의 LLM을 비슷한 코드로 사용할 수 있습니다.

### 실습 순서
1. OpenAI SDK 설치
2. OpenRouter API Key 입력
3. OpenRouter 클라이언트 생성
4. 무료 모델(`openrouter/free`) 호출
5. 다른 모델 호출
6. System / User 메시지 사용
7. Temperature 등 생성 옵션 조절

> **주의:** API Key는 비밀번호와 같은 정보입니다. 노트북 코드에 직접 적어서 공유하지 마세요.

## 1. 필요한 패키지 설치

Colab에는 여러 Python 패키지가 미리 설치되어 있지만, 최신 OpenAI SDK를 사용하기 위해 아래 셀을 먼저 실행합니다.

In [ ]:
# OpenAI Python SDK 설치
%pip -q install -U openai

## 2. Colab Secrets에서 API key 불러오기

아래 코드는 여러분이 Secrets에 저장한 `OPENAI_API_KEY`를 불러와 OpenAI 클라이언트를 만듭니다.

**수정할 부분은 없습니다.** 그대로 실행하세요.

In [ ]:
from google.colab import userdata

# Colab Secrets에서 개인 API 키를 안전하게 불러옵니다.
api_key = userdata.get("OPENROUTER_API_KEY")

## 3. OpenRouter에 연결되는 클라이언트 생성

OpenAI SDK의 `OpenAI()` 객체를 사용하되, `base_url`을 OpenRouter API 주소로 지정합니다.

이 설정이 핵심입니다.

In [ ]:
from openai import OpenAI

# OpenAI SDK가 OpenRouter 서버로 요청을 보내도록 설정
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

print("OpenRouter 클라이언트가 준비되었습니다.")

## 4. 무료 모델을 이용한 첫 번째 질문

`openrouter/free`는 OpenRouter가 현재 이용 가능한 무료 모델 중 요청에 적합한 모델을 자동으로 선택하는 라우터입니다.

따라서 수업에서 특정 유료 모델을 지정하지 않고 API 호출을 연습하기에 편리합니다.

> 무료 모델의 종류와 이용 가능 여부는 시점에 따라 달라질 수 있습니다.

In [ ]:
# 사용할 모델 지정
model = "openrouter/free" # 무료 모델을 OpenRouter가 알라서 지정 
###### 다른 모델 사용해 보기 ###########
# model="openai/gpt-5"
# model="deepseek/deepseek-v3.2"
# model="nvidia/nemotron-3-ultra-550b-a55b:free"


# 사용자 질문 작성
question = "생성형 AI가 무엇인지 대학생이 이해하기 쉽게 세 문장으로 설명해 주세요."

# OpenRouter를 통해 LLM에 요청 전송
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": question}
    ]
)

# 생성된 답변 출력
print(response.choices[0].message.content)

## 5. `system` 메시지와 `user` 메시지 사용

- `system`: 모델이 어떤 역할과 방식으로 답해야 하는지 지정
- `user`: 사용자가 실제로 입력하는 질문이나 요청

아래에서는 모델에게 'LLM 강사' 역할을 부여합니다.

In [ ]:
# system 메시지로 모델의 역할과 답변 방식을 지정
messages = [
    {
        "role": "system",
        "content": "당신은 거대언어모형을 처음 배우는 대학생에게 개념을 쉽게 설명하는 강사입니다."
    },
    {
        "role": "user",
        "content": "LLM에서 토큰(token)이 무엇인지 예를 들어 설명해 주세요."
    }
]

response = client.chat.completions.create(
    model="openrouter/free",
    messages=messages
)

print(response.choices[0].message.content)

## 6. Temperature 조절하기

`temperature`는 일반적으로 답변 생성의 다양성을 조절할 때 사용합니다.

- 낮은 값: 상대적으로 일관되고 보수적인 출력
- 높은 값: 상대적으로 다양한 출력

단, **지원 범위와 실제 작동 방식은 모델마다 다를 수 있습니다.**

In [ ]:
prompt = "생성형 AI를 활용한 커뮤니케이션 연구 주제를 5개 제안해 주세요."

# temperature를 비교하기 위한 함수
def ask_llm(temperature):
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content

print("===== 낮은 Temperature =====")
print(ask_llm(0.2))

print("\n===== 높은 Temperature =====")
print(ask_llm(1.0))

## 7. 최대 출력 길이 제한하기

`max_tokens`를 지정하면 모델이 생성하는 답변의 최대 토큰 수를 제한할 수 있습니다.

모델에 따라 지원되는 파라미터나 권장 방식이 다를 수 있으므로 실제 프로젝트에서는 해당 모델의 OpenRouter 페이지를 확인하는 것이 좋습니다.

In [ ]:
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": "Transformer의 self-attention을 간단하게 설명해 주세요."
        }
    ],
    max_tokens=200,
)

print(response.choices[0].message.content)

## 8. 특정 모델 사용하기

OpenRouter에서는 여러 회사의 모델을 같은 방식으로 호출할 수 있습니다.

모델을 바꾸려면 일반적으로 `model=`에 지정된 **모델 ID**만 변경하면 됩니다.

예를 들어 OpenRouter의 Models 페이지에서 원하는 모델을 선택한 뒤 표시된 모델 ID를 아래의 `MODEL_ID`에 입력할 수 있습니다.

> 유료 모델을 선택하면 OpenRouter 계정의 크레딧이 사용될 수 있습니다.

In [ ]:
# 기본값은 무료 모델 라우터로 설정
# 특정 모델을 사용하려면 아래 문자열을 OpenRouter의 모델 ID로 변경
MODEL_ID = "openrouter/free"

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": "RAG가 무엇인지 한 문단으로 설명해 주세요."
        }
    ]
)

print(response.choices[0].message.content)

## 9. 여러 질문을 순서대로 보내기

아래 예제는 여러 질문을 반복문으로 처리합니다.

각 요청은 독립적으로 전송되므로 이전 질문의 대화 맥락은 자동으로 이어지지 않습니다.

In [ ]:
questions = [
    "LLM이 무엇인가요?",
    "RAG가 무엇인가요?",
    "AI Agent가 무엇인가요?",
]

# 질문 목록을 하나씩 OpenRouter에 전송
for question in questions:
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {"role": "user", "content": question}
        ]
    )

    print(f"질문: {question}")
    print(f"답변: {response.choices[0].message.content}")
    print("-" * 60)

## 10. 대화 맥락을 유지하는 간단한 예제

LLM API 자체가 이전 호출 내용을 자동으로 기억하는 것은 아닙니다.

이전 `user`와 `assistant` 메시지를 `messages` 목록에 계속 추가하여 다시 전송하면 대화 맥락을 유지할 수 있습니다.

In [ ]:
# 대화 내용을 저장할 리스트
conversation = [
    {
        "role": "system",
        "content": "당신은 AI를 처음 공부하는 학생을 돕는 친절한 강사입니다."
    }
]

# 첫 번째 질문 추가
conversation.append({"role": "user", "content": "RAG가 무엇인가요?"})

response1 = client.chat.completions.create(
    model="openrouter/free",
    messages=conversation
)

answer1 = response1.choices[0].message.content
print("첫 번째 답변:\n", answer1)

# 모델의 첫 번째 답변을 대화 기록에 추가
conversation.append({"role": "assistant", "content": answer1})

# 앞선 답변을 전제로 후속 질문 추가
conversation.append({"role": "user", "content": "방금 설명한 내용을 한 문장으로 다시 요약해 주세요."})

response2 = client.chat.completions.create(
    model="openrouter/free",
    messages=conversation
)

print("\n두 번째 답변:\n", response2.choices[0].message.content)

## 핵심 정리

OpenRouter에서 OpenAI SDK를 사용할 때 핵심 코드는 다음과 같습니다.

```python
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {"role": "user", "content": "질문을 입력하세요."}
    ]
)

print(response.choices[0].message.content)
```

즉, **OpenAI SDK를 사용하면서 `base_url`을 OpenRouter로 지정하고 원하는 OpenRouter 모델 ID를 선택하는 것**이 핵심입니다.